# 08 - Train Online with GRPO (branched rollouts)

This notebook puts **Group Relative Policy Optimization** on paper in Mouse Core terms.

GRPO is PPO's clipped policy update **without a critic**. Instead of GAE from a value head, advantages come from comparing **G stochastic completions** that share the same start:

1. Grow a **trunk** trajectory (env + context of length `L`).
2. At chosen context lengths, **fork**: `deepcopy(env)` × G and copy `context[:L]`.
3. From each fork, sample a different completion with the stochastic policy.
4. Score the G completions (here: sum of rewards on the branch suffix).
5. Z-score those scores inside the group → one advantage per branch.
6. Stamp advantages onto completion steps, train with `GrpoObjective`.

```text
trunk context length L
        │
        ├─ branch 0  ~~~stochastic~~~→  return r0
        ├─ branch 1  ~~~stochastic~~~→  return r1
        ├─ ...
        └─ branch G-1 ~~~~~~~~~~~~~~~→  return r_{G-1}

        A_i = (r_i - mean(r)) / std(r)     # group_relative_advantages
```

Forks happen at **many** `L` values (not only task start), so the policy is trained under short and long in-context histories.

Compared to `07_train_online_ppo.ipynb`: policy head only, no value head; collection is branched rather than a single on-policy stream.

Build trunk envs and a separate **eval** group first. Each of `NUM_CYCLES` cycles grows trunks for `ROLLOUT_STEPS`, trains for `GRPO_EPOCHS` × `TRAIN_STEPS`, then scores eval with `EVAL_STEPS` tasks.


In [ ]:
import copy
from collections import deque

import torch
import numpy as np

import procedural_frozenlake  # noqa: F401 — registers Procedural-FrozenLake-v1
from mouse_gym import EnvConfig, make_env, make_group_env
from mouse_core.models.kv_policy import (
    cache_needs_rebuild,
    rebuild_starts,
    resolve_cache_bounds,
)

from mouse_core.data import DataLoader, Datastore
from mouse_core.models import Model, preferred_dtype, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import NumericEmbedder
from mouse_core.models.heads import DiscreteActionHead
from mouse_core.objectives import (
    GrpoObjective,
    batch_field,
    group_relative_advantages,
    sample_discrete_action,
)


MODEL_ID = "mouse-example-model-grpo"
MAX_ACTIONS = 4
MAX_OBS_DISCRETE = 64
MAX_EPISODE_STEPS = 30
EPISODES_PER_TASK = 20

# Small trunk count keeps forking readable; scale up later like the PPO notebook.
NUM_TRUNK_ENVS = 2
GROUP_SIZE = 4                                # G completions per fork
BRANCH_EVERY = 32                             # fork whenever trunk context length is a multiple of this
BRANCH_HORIZON = 48                           # max new steps per branch (also stops on task boundary)
MAX_CONTEXT = 256                             # truncate trunk / branch context fed to the model

SEQUENCE_LENGTH = 128
BATCH_SIZE = 4

NUM_CYCLES = 40                               # outer rollout/train/eval cycles
ROLLOUT_STEPS = 128                           # trunk growth steps per cycle (passed to run_rollout)
TRAIN_STEPS = 25                              # optimizer updates per GRPO epoch (passed to run_train)
GRPO_EPOCHS = 2                               # run_train calls per cycle (2 * 25 = 50 updates)
EVAL_STEPS = 1                                # tasks per eval env per cycle (passed to run_eval)

EVAL_NUM_ENVS = 4
EVAL_SEED_OFFSET = 1_000_000                  # held-out env seed stream (far from train)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Build Environment

Each trunk is a **single** `mouse_gym` env (not a big `GroupEnv`). Forking uses `copy.deepcopy(env)` so every branch starts from the same physical state as the trunk at length `L`, then diverges through stochastic actions.

Also build a separate **eval** `GroupEnv` (seed stream offset by `EVAL_SEED_OFFSET`) for greedy task scores between cycles — eval stays grouped even though trunks are single-env.


In [ ]:
def make_trunk_env(*, i: int):
    return make_env(EnvConfig(id='Procedural-FrozenLake-v1', name=f'proc_frozenlake_grpo_{i}', reset_seed=i, episodes_per_task=EPISODES_PER_TASK, task_reset_options={'regenerate_map': True}, kwargs={'width': 8, 'height': 8, 'max_episode_steps': MAX_EPISODE_STEPS, 'map_seed': i, 'slippery_success_rate': 1.0, 'permute_obs': True, 'permute_actions': True}))
trunk_envs = [make_trunk_env(i=i) for i in range(NUM_TRUNK_ENVS)]


In [ ]:
eval_configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"eval_frozenlake_{i}",
        reset_seed=i + EVAL_SEED_OFFSET,
        episodes_per_task=EPISODES_PER_TASK,
        task_reset_options={"regenerate_map": True},
        kwargs={
            "width": 8,
            "height": 8,
            "max_episode_steps": MAX_EPISODE_STEPS,
            "map_seed": i + EVAL_SEED_OFFSET,
            "slippery_success_rate": 1.0,
            "permute_obs": True,
            "permute_actions": True,
        },
    )
    for i in range(EVAL_NUM_ENVS)
]

eval_env = make_group_env(eval_configs)


## Model

Policy only: `DiscreteActionHead` → `predictions["action"]`. No value head — GRPO's baseline is the group mean reward.


In [ ]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")

encoder = NumericEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {"field": "action", "type": "discrete", "vocab_size": MAX_ACTIONS, "std": 0.02, "tokens": 1},
        {"field": "observation", "type": "discrete", "vocab_size": MAX_OBS_DISCRETE, "std": 0.02, "tokens": 1},
        {"field": "reward", "type": "rff", "std": 0.02, "in_min": 0.01, "in_max": 100.0, "tokens": 1},
        {"field": "done", "type": "discrete", "vocab_size": 5, "std": 0.02, "tokens": 1},
    ],
)

policy_head = DiscreteActionHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(
    encoder=encoder,
    backbone=backbone,
    heads=policy_head,  # single head → action_head="action"
).train().to(device=device, dtype=preferred_dtype(device))
model.backbone.model.compile()
print(model)


## Branching helpers

- `act_once` — stochastic action + behavior log-prob from the current context.
- `rollout_branch` — from a forked env + shared prefix, sample up to `BRANCH_HORIZON` steps (or until task done codes 3/4).
- `fork_group` — deepcopy env × G, copy context, roll G completions, return rows + scalar returns.

Shared **prefix** rows get `advantage = 0` at stamp time; **suffix** rows get the group-relative `A_i`. Training still feeds prefix+suffix so the transformer sees the same conditioning as at fork time.


In [ ]:
def _truncate(*, rows: list[dict], max_len: int) -> list[dict]:
    return rows[-max_len:] if len(rows) > max_len else list(rows)

@torch.no_grad()
def act_once(*, model: Model, context: list[dict]) -> tuple[dict, float]:
    """Sample one action from context; return (input_dict, old_log_prob)."""
    if not context:
        raise ValueError('act_once requires a non-empty context.')
    pending = [_truncate(rows=context, max_len=MAX_CONTEXT)]
    predictions, _, _ = model(pending, use_cache=False)
    logits = predictions['action'][:, -1]
    actions, log_probs = sample_discrete_action(logits=logits, num_actions=MAX_ACTIONS)
    return ({'action': actions[0].cpu().numpy()}, float(log_probs[0].item()))

def rollout_branch(*, model: Model, env, prefix: list[dict], horizon: int=BRANCH_HORIZON) -> tuple[list[dict], float]:
    """Stochastic completion from a shared prefix. Returns (suffix_rows, return)."""
    context = list(prefix)
    suffix: list[dict] = []
    ret = 0.0
    model.eval()
    for _ in range(horizon):
        inputs, log_prob = act_once(model=model, context=context)
        out = env.step(inputs)
        row = {**inputs, **out, 'old_log_prob': log_prob}
        row.pop('info', None)
        suffix.append(row)
        context.append(row)
        ret += float(row['reward'])
        if int(row['done']) in (3, 4):
            break
    return (suffix, ret)

def fork_group(*, model: Model, env, prefix: list[dict], group_size: int=GROUP_SIZE) -> tuple[list[list[dict]], list[float]]:
    """Fork env+context into G stochastic completions.

    Returns parallel lists: per-branch full sequences (prefix + suffix) and
    per-branch scalar returns (suffix reward sum only).
    """
    branches: list[list[dict]] = []
    returns: list[float] = []
    for _ in range(group_size):
        env_g = copy.deepcopy(env)
        suffix, ret = rollout_branch(model=model, env=env_g, prefix=prefix)
        branches.append([dict(r) for r in prefix] + suffix)
        returns.append(ret)
        env_g.close()
    return (branches, returns)

def stamp_advantages_with_prefix_len(*, branches: list[list[dict]], returns: list[float], prefix_len: int) -> list[list[dict]]:
    adv = group_relative_advantages(rewards=torch.tensor(returns, dtype=torch.float32))
    out: list[list[dict]] = []
    for g, rows in enumerate(branches):
        a = float(adv[g].item())
        new_rows = []
        for t, row in enumerate(rows):
            r = dict(row)
            r['advantage'] = 0.0 if t < prefix_len else a
            r.setdefault('old_log_prob', 0.0)
            new_rows.append(r)
        out.append(new_rows)
    return out


## Evaluation Phase

`run_eval` rolls out the current policy greedily on `eval_env`. It does not append to train replay. Row **contexts persist** across calls so evaluation continues mid-task; the **KV cache does not** and is rebuilt from those contexts at the start of each call. Each outer cycle passes `num_steps=EVAL_STEPS` (tasks per eval env).


In [ ]:
eval_contexts = [[] for _ in eval_env.names]

def run_eval(
    *,
    model,
    env,
    contexts: list,
    num_steps: int,
    max_cache: int = 512,
    start_cache=None,
    temperature: float = 0.0,
):
    """Greedy task-budget eval on a GroupEnv (does not append to train replay).

    ``num_steps`` is the number of tasks to complete per env stream.
    ``contexts`` persist across calls (mutated in place); the KV cache does not
    and is rebuilt from ``contexts`` at the start of each call.
    """
    model.eval()
    max_cache, start_cache = resolve_cache_bounds(max_cache, start_cache)
    n = len(env.names)
    if len(contexts) != n:
        raise ValueError(f"contexts has {len(contexts)} streams but env has {n}")
    kv_cache = None
    cached_starts = np.zeros(n, dtype=np.int64)
    cached_ends = np.zeros(n, dtype=np.int64)
    context_start = np.zeros(n, dtype=np.int64)
    eval_task_scores = [[] for _ in range(n)]
    inputs = None
    outputs = None
    env.metrics.clear()
    num_actions = env.action_space.spaces[0].n

    def _act_from_contexts(*, rebuild: bool) -> list[dict]:
        nonlocal kv_cache, cached_starts, cached_ends
        ends = np.array([len(c) for c in contexts], dtype=np.int64)
        need_rebuild = rebuild or cache_needs_rebuild(
            has_cache=kv_cache is not None,
            cached_starts=cached_starts,
            cached_ends=cached_ends,
            ends=ends,
            context_start=context_start,
            max_cache=max_cache,
            batch_complete=True,
        )
        with torch.no_grad():
            if need_rebuild:
                starts = rebuild_starts(
                    ends=ends,
                    context_start=context_start,
                    start_cache=start_cache,
                    max_cache=max_cache,
                )
                batch = [contexts[i][int(starts[i]) : int(ends[i])] for i in range(n)]
                predictions, _, kv_cache = model(batch, use_cache=True)
                cached_starts = starts
                cached_ends = ends.copy()
            else:
                batch = [
                    contexts[i][int(cached_ends[i]) : int(ends[i])] for i in range(n)
                ]
                predictions, _, kv_cache = model(batch, cache=kv_cache, use_cache=True)
                cached_ends = ends.copy()
            actions = model.get_action(
                predictions, temperature=temperature, num_actions=num_actions
            )
        random_inputs = env.sample_random_input()
        return [
            {"action": action} if contexts[i] else random_inputs[i]
            for i, action in enumerate(actions.cpu().numpy())
        ]

    while min(len(scores) for scores in eval_task_scores) < num_steps:
        if outputs is None:
            # Start of this call: rebuild KV from persisted contexts (or random if empty).
            if any(contexts):
                inputs = _act_from_contexts(rebuild=True)
            else:
                inputs = env.sample_random_input()
        else:
            task_ended = False
            for i, (inp, out) in enumerate(zip(inputs, outputs)):
                row = {**inp, **out}
                row.pop("info", None)
                contexts[i].append(row)
                if int(row["done"]) >= 3:
                    contexts[i] = []
                    context_start[i] = 0
                    task_ended = True
            inputs = _act_from_contexts(rebuild=task_ended)
        outputs = env.step(inputs)
        for i, out in enumerate(outputs):
            if int(out["done"]) >= 3:
                eval_task_scores[i].append(float(env.metrics.task_cum_rewards[i][-1]))

    scores_per_env = [scores[:num_steps] for scores in eval_task_scores]
    flat_scores = [s for scores in scores_per_env for s in scores]
    mean_task_score = (
        sum(flat_scores) / len(flat_scores) if flat_scores else float("nan")
    )
    return {
        "mean_task_score": mean_task_score,
        "task_scores": flat_scores,
        "scores_per_env": scores_per_env,
        "n_tasks": len(flat_scores),
    }


## Rollout Phase — trunk growth + forks at many `L`

Each cycle:

1. Advance every trunk one step (stochastic), appending to its context deque.
2. When `len(context)` is a multiple of `BRANCH_EVERY` (and at least 1), fork a GRPO group at that `L`.
3. Append stamped branch sequences into on-policy `Datastore`s for training.

Trunk contexts keep growing (bounded by `MAX_CONTEXT`) so later forks see longer histories.

Row **contexts persist** across calls; the **KV cache does not** and is rebuilt from those contexts at the start of each call.


In [ ]:
trunk_contexts: list[deque] = [deque(maxlen=MAX_CONTEXT) for _ in trunk_envs]
stores: list[Datastore] = []

def ensure_trunk_seed(*, model: Model, env, context: deque) -> None:
    """Take a random step if the trunk has no context yet."""
    if context:
        return
    inputs = env.sample_random_input()
    out = env.step(inputs)
    row = {**inputs, **out, 'old_log_prob': 0.0, 'advantage': 0.0}
    row.pop('info', None)
    context.append(row)


In [ ]:
def run_rollout(*, model: Model, num_steps: int) -> tuple[int, int]:
    """Grow trunks for ``num_steps``, fork at selected L, fill on-policy stores.

    Trunk contexts persist across calls. Returns (forks, branches).
    """
    global stores
    model.eval()
    stores = []
    n_forks = 0
    n_branches = 0
    for step in range(num_steps):
        for env, context in zip(trunk_envs, trunk_contexts):
            ensure_trunk_seed(model=model, env=env, context=context)
            inputs, log_prob = act_once(model=model, context=list(context))
            out = env.step(inputs)
            row = {**inputs, **out, 'old_log_prob': log_prob, 'advantage': 0.0}
            row.pop('info', None)
            context.append(row)
            L = len(context)
            if L > 0 and L % BRANCH_EVERY == 0:
                prefix = list(context)
                branches, returns = fork_group(group_size=GROUP_SIZE, model=model, env=env, prefix=prefix)
                stamped = stamp_advantages_with_prefix_len(prefix_len=L, branches=branches, returns=returns)
                for seq in stamped:
                    store = Datastore(name=f'grpo_branch_{n_branches}')
                    for r in seq:
                        store.append(r)
                    stores.append(store)
                    n_branches += 1
                n_forks += 1
    return (n_forks, n_branches)


## Training Phase

Each cycle builds a fresh `DataLoader` over that cycle's branch stores. Inject both `old_log_prob` and `advantage` from the sampled rows (they are not encoder modalities). No `polyak_update` — there is no target / value net.


In [ ]:
objective = GrpoObjective(clip_eps=0.2, ent_coef=0.01, num_actions=MAX_ACTIONS)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1.0e-5,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1.0e-8,
)

def run_train(
    *,
    model: Model,
    optimizer: torch.optim.Optimizer,
    objective: GrpoObjective,
    stores: list[Datastore],
    num_steps: int,
) -> tuple[torch.Tensor | None, dict[str, float]]:
    """Build a loader over this cycle's branch stores and run ``num_steps`` GRPO updates."""
    if not stores or sum(len(s) for s in stores) == 0:
        return None, {}
    model.train()
    loader = DataLoader(stores=stores, sequence_length=SEQUENCE_LENGTH, batch_size=min(BATCH_SIZE, len(stores)), weight_mode='per_step', num_workers=0)
    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    try:
        for _ in range(num_steps):
            batch = loader.next_batch()
            predictions, objective_data, _ = model(batch)
            objective_data["old_log_prob"] = batch_field(batch=batch, key='old_log_prob', device=objective_data.device)
            objective_data["advantage"] = batch_field(batch=batch, key='advantage', device=objective_data.device)
            loss, metrics = objective(objective_data, predictions)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    finally:
        loader.close()
    return loss, metrics


## Run

Each of `NUM_CYCLES` cycles collects forks over `ROLLOUT_STEPS` of trunk growth, runs `GRPO_EPOCHS` × `run_train(num_steps=TRAIN_STEPS)`, clears branch stores, then scores held-out tasks with `run_eval(num_steps=EVAL_STEPS)`.


In [ ]:
grad_steps = 0
for cycle in range(NUM_CYCLES):
    n_forks, n_branches = run_rollout(model=model, num_steps=ROLLOUT_STEPS)
    print(f"cycle={cycle} rollout  forks={n_forks}  branches={n_branches}")
    if n_branches == 0:
        continue
    for _ in range(GRPO_EPOCHS):
        loss, metrics = run_train(model=model, optimizer=optimizer, objective=objective, stores=stores, num_steps=TRAIN_STEPS)
        grad_steps += TRAIN_STEPS
    print(f"cycle={cycle} train    loss={loss.item():.4f}  policy={metrics['policy_loss']:.4f}  grad_step={grad_steps}")
    stores.clear()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    stats = run_eval(num_steps=EVAL_STEPS, max_cache=SEQUENCE_LENGTH, model=model, env=eval_env, contexts=eval_contexts)
    print(f"cycle={cycle} eval     mean_task_score={stats['mean_task_score']:.2f}/{EPISODES_PER_TASK}  n_tasks={stats['n_tasks']}")
for env in trunk_envs:
    env.close()
eval_env.close()
print(f'GRPO demo finished ({grad_steps} optimizer steps, {NUM_CYCLES} cycles).')


## Design notes (why this shape)

| Piece | Choice in this notebook |
|-------|-------------------------|
| Group | `G = GROUP_SIZE` deepcopy'd envs + identical `context[:L]` |
| Divergence | `sample_discrete_action` (stochastic policy) |
| Where to fork | every `BRANCH_EVERY` trunk steps → many `L` |
| Score | sum of rewards on the **suffix** only |
| Advantage | `group_relative_advantages(returns)` broadcast on suffix; `0` on prefix |
| Loss | `GrpoObjective` clipped surrogate + entropy (no critic) |

**Still open / scale-up knobs:** fork at task boundaries only; longer `BRANCH_HORIZON`; KL-to-reference term; caching KV across trunk steps; forking from a `GroupEnv` of many trunks in parallel.

## Push (optional)


In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model=model, repo_id=MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")
